# Getting Started with IDC and MONAI

Copyright 2026 Imaging Data Commons

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at http://www.apache.org/licenses/LICENSE-2.0

---

This tutorial introduces how to use data from the [NCI Imaging Data Commons (IDC)](https://portal.imaging.datacommons.cancer.gov/) with [MONAI](https://monai.io/).

## What You'll Learn

1. Query IDC to find relevant imaging data
2. Download DICOM data from IDC
3. Load IDC data into MONAI using transforms
4. Create a MONAI-compatible dataset

## Prerequisites

- Basic Python knowledge
- Familiarity with PyTorch DataLoaders
- No authentication required for IDC data access

## Setup environment

In [ ]:
!pip install -q monai idc-index

## Setup imports

In [ ]:
import os
import tempfile

from idc_index import IDCClient
import matplotlib.pyplot as plt

from monai.transforms import (
    Compose,
    LoadImaged,
    EnsureChannelFirstd,
    Orientationd,
    Spacingd,
    ScaleIntensityRanged,
)
from monai.data import Dataset, DataLoader
import monai

monai.config.print_config()

## Introduction to IDC

The Imaging Data Commons (IDC) is a cloud-based repository of publicly available cancer imaging data. Key features:

- **50+ TB** of radiology and pathology images
- **No authentication** required for data access
- Data organized into **collections** by disease/study
- Includes **AI-generated annotations** and segmentations
- All data is in **DICOM format**

We'll use the `idc-index` package to query and download data.

## 1. Initialize IDC Client and Explore Data

In [ ]:
# Initialize the IDC client
client = IDCClient()

# Check IDC version
print(f"IDC data version: {client.get_idc_version()}")

In [ ]:
# Get overview of IDC data
stats = client.sql_query("""
    SELECT   
        COUNT(DISTINCT collection_id) as collections,
        COUNT(DISTINCT PatientID) as patients,
        COUNT(DISTINCT SeriesInstanceUID) as series,
        ROUND(SUM(series_size_MB)/1000000, 2) as size_TB
    FROM index
""")
print("IDC Data Overview:")
print(stats)

## 2. Query Collections

IDC data is organized into collections. Let's explore what's available using the `collections_index`.

In [ ]:
# Fetch and query collections_index for collection metadata
client.fetch_index("collections_index")

# Find lung cancer CT collections
lung_collections = client.sql_query("""
    SELECT 
        collection_id,
        Subjects,
        CancerTypes,
        Modalities
    FROM collections_index
    WHERE CancerTypes LIKE '%Lung%'
        AND Modalities LIKE '%CT%'
        AND Species = 'Human'
    ORDER BY Subjects DESC
    LIMIT 10
""")

print("Lung Cancer CT Collections:")
print(lung_collections.to_string(index=False))

## 3. Query for Specific Series

Now let's query for specific CT series from a collection. We'll use a small collection for this demo.

In [ ]:
# Query for chest CT series from a small collection
# Using rider_pilot which is a small test collection (~1GB total)
series_df = client.sql_query("""
    SELECT 
        SeriesInstanceUID,
        PatientID,
        Modality,
        SeriesDescription,
        ROUND(series_size_MB, 2) as size_mb
    FROM index
    WHERE collection_id = 'rider_pilot'
        AND Modality = 'CT'
    LIMIT 3
""")

print(f"Found {len(series_df)} CT series:")
print(series_df.to_string(index=False))

## 4. Download Data

Download the DICOM series to a local directory.

In [ ]:
# Create a temporary directory for downloads
data_dir = tempfile.mkdtemp(prefix="idc_monai_")
print(f"Download directory: {data_dir}")

# Download the series
series_uids = list(series_df['SeriesInstanceUID'])
print(f"\nDownloading {len(series_uids)} series...")

client.download_from_selection(
    seriesInstanceUID=series_uids,
    downloadDir=data_dir,
    dirTemplate="%SeriesInstanceUID"  # Organize by series UID
)

print("Download complete!")

In [ ]:
# Verify downloaded files
for uid in series_uids:
    series_path = os.path.join(data_dir, uid)
    if os.path.exists(series_path):
        files = os.listdir(series_path)
        print(f"{uid[:30]}...: {len(files)} DICOM files")

## 5. Load Data with MONAI Transforms

MONAI's `LoadImaged` with ITKReader can directly load DICOM series from a directory. No format conversion needed!

In [ ]:
# Define MONAI transforms for CT preprocessing
transforms = Compose([
    # Load DICOM series (ITKReader handles directories automatically)
    LoadImaged(keys=["image"]),
    # Ensure channel-first format (C, H, W, D)
    EnsureChannelFirstd(keys=["image"]),
    # Standardize orientation to RAS
    Orientationd(keys=["image"], axcodes="RAS"),
    # Resample to isotropic spacing
    Spacingd(keys=["image"], pixdim=(1.5, 1.5, 2.0), mode="bilinear"),
    # Apply CT windowing (soft tissue window)
    ScaleIntensityRanged(
        keys=["image"],
        a_min=-175,  # Lower HU bound
        a_max=250,   # Upper HU bound  
        b_min=0.0,
        b_max=1.0,
        clip=True,
    ),
])

In [ ]:
# Prepare data dictionaries for MONAI Dataset
data_dicts = [
    {"image": os.path.join(data_dir, uid)}
    for uid in series_uids
]

print(f"Created {len(data_dicts)} data dictionaries")
print(f"Example: {data_dicts[0]}")

In [ ]:
# Create MONAI Dataset
dataset = Dataset(data=data_dicts, transform=transforms)

print(f"Dataset size: {len(dataset)}")

In [ ]:
# Load and inspect first sample
sample = dataset[0]

print(f"Image shape: {sample['image'].shape}")
print(f"Image dtype: {sample['image'].dtype}")
print(f"Value range: [{sample['image'].min():.3f}, {sample['image'].max():.3f}]")

## 6. Visualize the Data

In [ ]:
# Visualize slices from the CT volume
image = sample['image'][0]  # Remove channel dimension

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Axial slice (middle)
z_mid = image.shape[2] // 2
axes[0].imshow(image[:, :, z_mid].T, cmap='gray', origin='lower')
axes[0].set_title(f'Axial (slice {z_mid})')
axes[0].axis('off')

# Coronal slice (middle)
y_mid = image.shape[1] // 2
axes[1].imshow(image[:, y_mid, :].T, cmap='gray', origin='lower')
axes[1].set_title(f'Coronal (slice {y_mid})')
axes[1].axis('off')

# Sagittal slice (middle)
x_mid = image.shape[0] // 2
axes[2].imshow(image[x_mid, :, :].T, cmap='gray', origin='lower')
axes[2].set_title(f'Sagittal (slice {x_mid})')
axes[2].axis('off')

plt.suptitle('CT Volume from IDC', fontsize=14)
plt.tight_layout()
plt.show()

## 7. Create a DataLoader for Training

Now we can use the dataset with a PyTorch DataLoader for training.

In [ ]:
# Create DataLoader
dataloader = DataLoader(
    dataset,
    batch_size=1,  # Typically 1-2 for 3D medical images
    shuffle=True,
    num_workers=0,  # Set to 0 for this demo
)

# Iterate through one batch
for batch in dataloader:
    print(f"Batch image shape: {batch['image'].shape}")
    break

## 8. Check Data Licenses

Before using IDC data, always check the license. CC-BY allows commercial use, while CC-BY-NC restricts to non-commercial.

In [ ]:
# Check licenses for our downloaded series
uid_list = ", ".join(f"'{uid}'" for uid in series_uids)
licenses = client.sql_query(f"""
    SELECT SeriesInstanceUID, license_short_name
    FROM index
    WHERE SeriesInstanceUID IN ({uid_list})
""")

print("License Information:")
for _, row in licenses.iterrows():
    uid_short = row['SeriesInstanceUID'][:30]
    print(f"  {uid_short}...: {row['license_short_name']}")

## 9. Cleanup

In [ ]:
# Optionally cleanup downloaded data
# Uncomment to delete:
# import shutil
# shutil.rmtree(data_dir)
# print(f"Cleaned up {data_dir}")

print(f"Data saved at: {data_dir}")

## Summary

In this tutorial, you learned how to:

1. **Query IDC** using SQL with `idc-index` to find relevant imaging data
2. **Download DICOM** series to a local directory
3. **Load data** directly into MONAI using `LoadImaged` (no format conversion needed)
4. **Apply transforms** for CT preprocessing (orientation, spacing, intensity)
5. **Create datasets and dataloaders** for training

## Next Steps

- **Tutorial 2**: CT Segmentation with IDC data and paired annotations
- **Tutorial 3**: Working with IDC analysis results (AI segmentations)

## Resources

- [IDC Portal](https://portal.imaging.datacommons.cancer.gov/)
- [IDC Documentation](https://learn.canceridc.dev/)
- [MONAI Tutorials](https://github.com/Project-MONAI/tutorials)
- [idc-index GitHub](https://github.com/ImagingDataCommons/idc-index)